# Notebook 22 — Predictive Decompression Forecasting

**Repo:** `int_serialization_benchmark-rml`  
**Layer:** `rml_extension/notebooks/`

Notebook 21 made macro-route thresholds adaptive.

Notebook 22 forecasts decompression before fallback occurs.

Constraint view:
> recursive compression should expand before coherence collapse propagates through compressed macro routes.

## Goals

1. Load Notebook 21 adaptive threshold gating outputs when available.
2. Build a forecast target for future decompression.
3. Train a lightweight classifier.
4. Evaluate forecast probability, ROC AUC, confusion matrix, and feature importance.
5. Export CSV, JSON, Markdown report, and PNG figures.
6. Generate a Colab-downloadable output zip.

This notebook follows the prior repo notebook style:

- separate notebook sections,
- saved figures plus `plt.show()` rendering,
- generated outputs section in the report,
- `figures/` link style in the report.

In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split

cwd = Path.cwd()
candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/int_serialization_benchmark-rml"),
    Path("/content"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "rml_extension").exists() or (c / "configs").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RML_ROOT = REPO_ROOT / "rml_extension" if (REPO_ROOT / "rml_extension").exists() else REPO_ROOT

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"
REPORTS_DIR = RML_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("RML_ROOT:", RML_ROOT)

## Load Notebook 21 outputs

Uses:

```text
results/notebook21_adaptive_threshold_gating.csv
```

If that file is unavailable, this notebook creates a fallback synthetic routing stream.

In [ ]:
input_path = RESULTS_DIR / "notebook21_adaptive_threshold_gating.csv"

if input_path.exists():
    df = pd.read_csv(input_path)
    print("Loaded:", input_path)
else:
    print("Notebook 21 output not found; creating fallback synthetic routing stream.")
    rng = np.random.default_rng(42)
    N = 240
    windows = np.arange(N)
    macro_routes = rng.choice(
        ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"],
        size=N,
        p=[0.34, 0.12, 0.20, 0.18, 0.16],
    )

    macro_cgcs_score = (
        0.55
        + 0.08 * np.sin(np.linspace(0, 8 * np.pi, N))
        + rng.normal(0, 0.09, N)
    )
    macro_cgcs_score[105:145] += 0.12
    macro_cgcs_score = np.clip(macro_cgcs_score, 0.2, 0.82)

    rolling_stability = pd.Series(macro_cgcs_score).rolling(12, min_periods=1).mean()
    rolling_volatility = pd.Series(macro_cgcs_score).rolling(10, min_periods=1).std().fillna(0)
    rolling_switch_rate = (
        pd.Series(macro_routes)
        .ne(pd.Series(macro_routes).shift())
        .rolling(15, min_periods=1)
        .mean()
    )
    rolling_residual = 1 - rolling_stability + rolling_volatility
    rolling_pressure = (
        0.4 * rolling_switch_rate
        + 0.35 * rolling_volatility
        + 0.25 * rolling_residual
    )
    rolling_pressure = (
        (rolling_pressure - rolling_pressure.min())
        / (rolling_pressure.max() - rolling_pressure.min())
    )

    df = pd.DataFrame({
        "window_id": windows,
        "macro_route": macro_routes,
        "macro_cgcs_score": macro_cgcs_score,
        "rolling_stability": rolling_stability,
        "rolling_cgcs_std": rolling_volatility,
        "fixed_gated_switch_rate": rolling_switch_rate,
        "adaptive_gated_switch_rate": rolling_switch_rate * 0.85,
        "rolling_residual": rolling_residual,
        "rolling_pressure": rolling_pressure,
        "adaptive_gate": np.where(macro_cgcs_score > 0.70, "accepted", np.where(macro_cgcs_score > 0.50, "watch", "fallback")),
        "early_decompression_candidate": (macro_cgcs_score < 0.48) & (rolling_pressure > 0.50),
    })

df = df.sort_values(df.columns[0]).reset_index(drop=True)

if "window_id" not in df.columns:
    if "window" in df.columns:
        df["window_id"] = df["window"]
    else:
        df["window_id"] = np.arange(len(df))

if "macro_route" not in df.columns:
    df["macro_route"] = "macro_unknown"

if "macro_cgcs_score" not in df.columns:
    df["macro_cgcs_score"] = 0.5

df.head()

## Normalize forecast features

Notebook 22 uses the available Notebook 21 columns where possible.
Missing columns are reconstructed from the macro CGCS score and route-switch signals.

In [ ]:
def ensure_numeric(col, default=0.0):
    if col not in df.columns:
        df[col] = default
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)

ensure_numeric("macro_cgcs_score", 0.5)

if "rolling_stability" not in df.columns:
    df["rolling_stability"] = df["macro_cgcs_score"].rolling(12, min_periods=1).mean()
ensure_numeric("rolling_stability", 0.5)

if "rolling_cgcs_std" not in df.columns:
    df["rolling_cgcs_std"] = df["macro_cgcs_score"].rolling(10, min_periods=1).std().fillna(0)
ensure_numeric("rolling_cgcs_std", 0.0)

if "rolling_volatility" not in df.columns:
    df["rolling_volatility"] = df["rolling_cgcs_std"]
ensure_numeric("rolling_volatility", 0.0)

if "adaptive_gated_switch_rate" not in df.columns:
    route_changed = df["macro_route"].ne(df["macro_route"].shift()).fillna(False)
    df["adaptive_gated_switch_rate"] = route_changed.rolling(15, min_periods=1).mean()
ensure_numeric("adaptive_gated_switch_rate", 0.0)

if "rolling_switch_rate" not in df.columns:
    if "fixed_gated_switch_rate" in df.columns:
        df["rolling_switch_rate"] = df["fixed_gated_switch_rate"]
    else:
        df["rolling_switch_rate"] = df["adaptive_gated_switch_rate"]
ensure_numeric("rolling_switch_rate", 0.0)

if "rolling_residual" not in df.columns:
    if "macro_compression_residual" in df.columns:
        df["rolling_residual"] = pd.to_numeric(df["macro_compression_residual"], errors="coerce").fillna(0.0).rolling(12, min_periods=1).mean()
    else:
        df["rolling_residual"] = (1 - df["rolling_stability"] + df["rolling_volatility"]).clip(0, 1)
ensure_numeric("rolling_residual", 0.0)

if "rolling_pressure" not in df.columns:
    raw_pressure = (
        0.40 * df["rolling_switch_rate"]
        + 0.35 * df["rolling_volatility"]
        + 0.25 * df["rolling_residual"]
    )
    denom = raw_pressure.max() - raw_pressure.min()
    df["rolling_pressure"] = (raw_pressure - raw_pressure.min()) / denom if denom != 0 else 0.0
ensure_numeric("rolling_pressure", 0.0)

if "adaptive_gate" not in df.columns:
    df["adaptive_gate"] = np.where(
        df["macro_cgcs_score"] >= 0.70,
        "accepted",
        np.where(df["macro_cgcs_score"] >= 0.50, "watch", "fallback")
    )

if "early_decompression_candidate" not in df.columns:
    df["early_decompression_candidate"] = (
        (df["macro_cgcs_score"] < 0.48) &
        (df["rolling_pressure"] > 0.50)
    )

df[[
    "window_id",
    "macro_route",
    "macro_cgcs_score",
    "rolling_stability",
    "rolling_volatility",
    "rolling_switch_rate",
    "rolling_residual",
    "rolling_pressure",
    "adaptive_gate",
    "early_decompression_candidate",
]].head()

## Build future decompression target

The forecast target is whether decompression/fallback pressure appears within the next `forecast_horizon` windows.

In [ ]:
forecast_horizon = 5

decompression_event = (
    df["early_decompression_candidate"].astype(bool)
    | df["adaptive_gate"].eq("fallback")
    | ((df["macro_cgcs_score"] < 0.45) & (df["rolling_pressure"] > 0.55))
).astype(int)

future_decompression = (
    pd.Series(decompression_event)
    .rolling(forecast_horizon, min_periods=1)
    .max()
    .shift(-forecast_horizon)
    .fillna(0)
    .astype(int)
)

df["decompression_event"] = decompression_event
df["future_decompression"] = future_decompression

df[["window_id", "decompression_event", "future_decompression"]].head(12)

## Feature matrix

In [ ]:
route_encoding = {r: i for i, r in enumerate(sorted(df["macro_route"].astype(str).unique()))}
df["macro_route_id"] = df["macro_route"].astype(str).map(route_encoding)

feature_cols = [
    "macro_cgcs_score",
    "rolling_stability",
    "rolling_volatility",
    "rolling_switch_rate",
    "rolling_residual",
    "rolling_pressure",
    "macro_route_id",
]

X = df[feature_cols].copy()
y = df["future_decompression"].astype(int)

print("Positive target windows:", int(y.sum()), "of", len(y))
X.head()

## Train predictor

In [ ]:
# Stratify when both classes are present with enough samples.
stratify = y if y.nunique() == 2 and y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=stratify,
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
)
model.fit(X_train, y_train)

if len(model.classes_) == 2:
    class_one_index = list(model.classes_).index(1)
    probs = model.predict_proba(X)[:, class_one_index]
else:
    probs = np.zeros(len(df))

preds = (probs > 0.5).astype(int)

df["decompression_probability"] = probs
df["predicted_decompression"] = preds

df[["window_id", "future_decompression", "decompression_probability", "predicted_decompression"]].head()

## Evaluation

In [ ]:
if y.nunique() == 2:
    auc = roc_auc_score(y, probs)
    fpr, tpr, _ = roc_curve(y, probs)
else:
    auc = float("nan")
    fpr = np.array([0, 1])
    tpr = np.array([0, 1])

cm = confusion_matrix(y, preds, labels=[0, 1])
report_df = pd.DataFrame(
    classification_report(y, preds, labels=[0, 1], output_dict=True, zero_division=0)
).transpose()

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print("ROC AUC:", auc)
feature_importance

## Forecast transition matrix

In [ ]:
states = ["stable", "forecast"]
forecast_state = np.where(df["predicted_decompression"] == 1, "forecast", "stable")

transition_counts = pd.DataFrame(0, index=states, columns=states, dtype=float)
for i in range(len(forecast_state) - 1):
    transition_counts.loc[forecast_state[i], forecast_state[i + 1]] += 1

transition_probs = transition_counts.div(
    transition_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).fillna(0)

transition_probs

## Save output tables

In [ ]:
results_csv = RESULTS_DIR / "notebook22_predictive_decompression.csv"
results_json = RESULTS_DIR / "notebook22_predictive_decompression.json"
summary_csv = RESULTS_DIR / "notebook22_summary.csv"
importance_csv = RESULTS_DIR / "notebook22_feature_importance.csv"
transition_csv = RESULTS_DIR / "notebook22_forecast_transition_matrix.csv"
classification_csv = RESULTS_DIR / "notebook22_classification_report.csv"
confusion_csv = RESULTS_DIR / "notebook22_confusion_matrix.csv"

df.to_csv(results_csv, index=False)
df.to_json(results_json, orient="records", indent=2)

summary = {
    "windows": int(len(df)),
    "forecast_horizon": int(forecast_horizon),
    "forecast_positive_windows": int(preds.sum()),
    "actual_future_decompression_windows": int(y.sum()),
    "roc_auc": float(auc) if not np.isnan(auc) else None,
    "mean_probability": float(probs.mean()),
    "mean_pressure": float(df["rolling_pressure"].mean()),
    "mean_stability": float(df["rolling_stability"].mean()),
    "mean_switch_rate": float(df["rolling_switch_rate"].mean()),
}
summary_df = pd.DataFrame([summary])

summary_df.to_csv(summary_csv, index=False)
feature_importance.to_csv(importance_csv, index=False)
transition_probs.to_csv(transition_csv)
report_df.to_csv(classification_csv)
pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"]).to_csv(confusion_csv)

print("Saved:", results_csv)
print("Saved:", results_json)
print("Saved:", summary_csv)
print("Saved:", importance_csv)
print("Saved:", transition_csv)
print("Saved:", classification_csv)
print("Saved:", confusion_csv)

## Figure 1 — Probability timeline

In [ ]:
forecast_timeline_fig = FIGURES_DIR / "notebook22_probability_timeline.png"

plt.figure(figsize=(16, 6))
plt.plot(df["window_id"], probs, label="forecast probability")
plt.axhline(0.5, linestyle="--", label="forecast threshold")
plt.title("Predictive Decompression Forecasting: Probability Timeline")
plt.xlabel("Window")
plt.ylabel("Forecast probability")
plt.legend()
plt.tight_layout()
plt.savefig(forecast_timeline_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", forecast_timeline_fig)

## Figure 2 — Actual vs predicted

In [ ]:
forecast_vs_actual_fig = FIGURES_DIR / "notebook22_actual_vs_predicted.png"

plt.figure(figsize=(16, 6))
plt.plot(df["window_id"], y, label="actual future decompression")
plt.plot(df["window_id"], preds, label="predicted decompression")
plt.title("Predictive Decompression Forecasting: Actual vs Predicted")
plt.xlabel("Window")
plt.ylabel("Forecast state")
plt.legend()
plt.tight_layout()
plt.savefig(forecast_vs_actual_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", forecast_vs_actual_fig)

## Figure 3 — Feature importance

In [ ]:
importance_fig = FIGURES_DIR / "notebook22_feature_importance.png"

plt.figure(figsize=(10, 6))
plt.bar(feature_importance["feature"], feature_importance["importance"])
plt.xticks(rotation=35, ha="right")
plt.title("Predictive Decompression Forecasting: Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.savefig(importance_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", importance_fig)

## Figure 4 — Forecast transition matrix

In [ ]:
transition_fig = FIGURES_DIR / "notebook22_forecast_transition_matrix.png"

plt.figure(figsize=(6, 6))
plt.imshow(transition_probs.values, aspect="auto")
plt.xticks(range(len(states)), states, rotation=35, ha="right")
plt.yticks(range(len(states)), states)
plt.colorbar(label="Transition probability")
plt.title("Predictive Decompression Forecasting: Transition Matrix")
plt.xlabel("Next forecast state")
plt.ylabel("Current forecast state")
plt.tight_layout()
plt.savefig(transition_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", transition_fig)

## Figure 5 — ROC curve

In [ ]:
roc_fig = FIGURES_DIR / "notebook22_roc_curve.png"

plt.figure(figsize=(7, 7))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}" if not np.isnan(auc) else "AUC unavailable")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Predictive Decompression Forecasting: ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig(roc_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", roc_fig)

## Figure 6 — Pressure vs probability

In [ ]:
pressure_probability_fig = FIGURES_DIR / "notebook22_pressure_vs_probability.png"

plt.figure(figsize=(16, 6))
plt.plot(df["window_id"], df["rolling_pressure"], label="rolling pressure")
plt.plot(df["window_id"], probs, label="forecast probability")
plt.title("Predictive Decompression Forecasting: Pressure vs Probability")
plt.xlabel("Window")
plt.ylabel("Normalized score")
plt.legend()
plt.tight_layout()
plt.savefig(pressure_probability_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", pressure_probability_fig)

## Figure 7 — PCA route projection

In [ ]:
projection_fig = FIGURES_DIR / "notebook22_pca_projection.png"

pca = PCA(n_components=2)
coords = pca.fit_transform(X)

plt.figure(figsize=(8, 8))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=probs)
plt.colorbar(scatter, label="Forecast probability")
plt.title("Predictive Decompression Forecasting: PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig(projection_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Saved:", projection_fig)

## Full Markdown report

In [ ]:
report_path = REPORTS_DIR / "report_22_predictive_decompression_forecasting.md"

def rel(path):
    p = Path(path)
    try:
        return str(p.relative_to(RML_ROOT))
    except Exception:
        return str(p)

# Use figures/ link style in report, as requested.
results_csv_link = "results/notebook22_predictive_decompression.csv"
results_json_link = "results/notebook22_predictive_decompression.json"
summary_csv_link = "results/notebook22_summary.csv"
importance_csv_link = "results/notebook22_feature_importance.csv"
transition_csv_link = "results/notebook22_forecast_transition_matrix.csv"
classification_csv_link = "results/notebook22_classification_report.csv"
confusion_csv_link = "results/notebook22_confusion_matrix.csv"

forecast_timeline_link = "figures/notebook22_probability_timeline.png"
forecast_vs_actual_link = "figures/notebook22_actual_vs_predicted.png"
importance_fig_link = "figures/notebook22_feature_importance.png"
transition_fig_link = "figures/notebook22_forecast_transition_matrix.png"
roc_fig_link = "figures/notebook22_roc_curve.png"
pressure_probability_fig_link = "figures/notebook22_pressure_vs_probability.png"
projection_fig_link = "figures/notebook22_pca_projection.png"

report_lines = [
    "# Report 22 — Predictive Decompression Forecasting",
    "",
    "This report forecasts decompression events before fallback occurs.",
    "",
    "Constraint view:",
    "> recursive compression should expand before coherence collapse propagates through compressed macro routes.",
    "",
    "## Generated outputs",
    "",
    f'- Predictive decompression CSV: <a href="{results_csv_link}">`{results_csv_link}`</a>',
    f'- Predictive decompression JSON: <a href="{results_json_link}">`{results_json_link}`</a>',
    f'- Summary CSV: <a href="{summary_csv_link}">`{summary_csv_link}`</a>',
    f'- Feature importance CSV: <a href="{importance_csv_link}">`{importance_csv_link}`</a>',
    f'- Forecast transition matrix CSV: <a href="{transition_csv_link}">`{transition_csv_link}`</a>',
    f'- Classification report CSV: <a href="{classification_csv_link}">`{classification_csv_link}`</a>',
    f'- Confusion matrix CSV: <a href="{confusion_csv_link}">`{confusion_csv_link}`</a>',
    f'- Figure: <a href="{forecast_timeline_link}">`{forecast_timeline_link}`</a>',
    f'- Figure: <a href="{forecast_vs_actual_link}">`{forecast_vs_actual_link}`</a>',
    f'- Figure: <a href="{importance_fig_link}">`{importance_fig_link}`</a>',
    f'- Figure: <a href="{transition_fig_link}">`{transition_fig_link}`</a>',
    f'- Figure: <a href="{roc_fig_link}">`{roc_fig_link}`</a>',
    f'- Figure: <a href="{pressure_probability_fig_link}">`{pressure_probability_fig_link}`</a>',
    f'- Figure: <a href="{projection_fig_link}">`{projection_fig_link}`</a>',
    "",
    "## Summary",
    "",
    summary_df.to_markdown(index=False),
    "",
    "## Feature importance",
    "",
    feature_importance.to_markdown(index=False),
    "",
    "## Forecast transition probabilities",
    "",
    transition_probs.to_markdown(),
    "",
    "## Classification report",
    "",
    report_df.to_markdown(),
    "",
    "## Interpretation",
    "",
    "- Forecast probability estimates decompression risk before fallback emerges.",
    "- Rolling pressure, residual instability, and switch-rate volatility dominate decompression forecasting.",
    "- Stable compressed plateaus reduce decompression probability.",
    "- Pressure spikes increase forecast instability and future decompression likelihood.",
    "- Forecast transition structure reveals persistence between stable and unstable routing phases.",
    "",
    "## Next step",
    "",
    "Notebook 23 can build predictive constraint routing: route around forecasted decompression before fallback occurs.",
]

report_path.write_text("\n".join(report_lines))
print("Saved:", report_path)

## Optional: download output bundle in Colab

Uncomment the following cell if running in Google Colab.

In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook22_predictive_decompression_forecasting_outputs.zip"
# export_path = RML_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook22_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#         for p in folder.glob("report_22_*"):
#             zf.write(p, arcname=str(p.relative_to(RML_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))